# Phase 9 Worksheet — Enterprise Pipelines to ChromaDB
**Corrected in this version:** `ingest_pipeline()` now calls `embedder.embed_documents()` instead of looping `get_embedding(t, model=MODEL_JINA)` per item.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../../wrapper_fix"))  # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("../../inhouse_rag_capstone"))  # folder containing your real inhouse_llm.py

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=200):
    """Drop-in replacement for the old multimodal_chat() text-only calls --
    correctly routed per-model via get_chat_model(), unlike inhouse_llm.py's
    own chat()/multimodal_chat() which always hit the Qwen3-14B endpoint."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=200):
    """Drop-in replacement for multimodal_chat() WITH an image -- uses the
    corrected image_url content-block format, and an actual client for the
    vision model (inhouse_llm.py never created one)."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

import chromadb
client = chromadb.HttpClient(host="localhost", port=8000)  # adjust to your Chroma server
print("Setup OK")

## 1. A generic pipeline shape, parameterized by source type

In [ ]:
def ingest_pipeline(collection_name, items, id_fn, text_fn, metadata_fn):
    """items: list of source-specific records. id_fn/text_fn/metadata_fn: how to
    derive Chroma id/text/metadata from each record -- this is the one function
    every one of the 5 pipelines reuses, just with different extractor functions."""
    coll = client.get_or_create_collection(collection_name)
    ids = [id_fn(item) for item in items]
    texts = [text_fn(item) for item in items]
    metadatas = [metadata_fn(item) for item in items]
    embeddings = embedder.embed_documents(texts)
    coll.upsert(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)
    return coll

## 2. Swagger -> ChromaDB

In [ ]:
swagger_endpoints = [
    {"path": "/payments", "method": "GET", "summary": "List payments"},
    {"path": "/payments", "method": "POST", "summary": "Create a payment"},
]
swagger_coll = ingest_pipeline(
    "phase9_swagger",
    swagger_endpoints,
    id_fn=lambda e: f"{e['method']}_{e['path']}".replace("/", "_"),
    text_fn=lambda e: f"{e['method']} {e['path']}: {e['summary']}",
    metadata_fn=lambda e: {"path": e["path"], "method": e["method"]},
)
print("Swagger collection count:", swagger_coll.count())

## 3. Database -> ChromaDB (with the row-vs-description granularity decision)

In [ ]:
payments_rows = [{"id": 1, "service": "Payment", "status": "Failed"}, {"id": 2, "service": "Payment", "status": "OK"}]
db_coll = ingest_pipeline(
    "phase9_database",
    payments_rows,
    id_fn=lambda r: f"payment_{r['id']}",
    text_fn=lambda r: f"Transaction {r['id']}: service={r['service']}, status={r['status']}",
    metadata_fn=lambda r: {"table": "payments", "row_id": r["id"]},
)
print("DB collection count:", db_coll.count())

## 4. Wiki -> ChromaDB, WITH re-sync via upsert (this phase's teaser, solved)

In [ ]:
wiki_pages_v1 = [{"page_id": "auth_guide", "version": 1, "text": "Use API version v1 for authentication."}]
wiki_coll = ingest_pipeline(
    "phase9_wiki",
    wiki_pages_v1,
    id_fn=lambda p: p["page_id"],
    text_fn=lambda p: p["text"],
    metadata_fn=lambda p: {"version": p["version"]},
)
print("Before edit:", wiki_coll.get(ids=["auth_guide"])["documents"])

wiki_pages_v2 = [{"page_id": "auth_guide", "version": 2, "text": "Use API version v2 for authentication."}]
ingest_pipeline("phase9_wiki", wiki_pages_v2,
                id_fn=lambda p: p["page_id"], text_fn=lambda p: p["text"], metadata_fn=lambda p: {"version": p["version"]})
print("After re-sync (upsert, SAME id):", wiki_coll.get(ids=["auth_guide"])["documents"])
print("-> the stale v1 content was overwritten, not duplicated, because the id stayed the same")

## 5. Detecting orphaned entries (deleted source content)

In [ ]:
current_wiki_ids = {"auth_guide"}  # simulate: this page still exists
stale_check_ids = {"auth_guide", "deleted_page_old"}  # simulate: this one was in Chroma from a previous run

orphaned = stale_check_ids - current_wiki_ids
print("Orphaned ids that should be deleted from Chroma:", orphaned)
# wiki_coll.delete(ids=list(orphaned))  # uncomment to actually clean up

## Teaser exercise
Build the Logs -> ChromaDB pipeline using `ingest_pipeline`, combined with the sliding window chunking function from Phase 2's worksheet as the `text_fn` (group log lines into overlapping windows before embedding each window).